Notebook:

1. Generuje realistyczny zbiór danych dotyczący predykcji odejścia klientów – Churn w firmie telekomunikacyjnej/SaaS).

2. Pokazuje całą ścieżkę: 
- problem niezbalansowanych klas, 
- kalibracja progu ryzyka i krzywa ROC-AUC, 
- wizualizacja reguł pojedynczego drzewa i  modeli zespołowych (Random Forest oraz Gradient Boosting) z analizą ważności cech.

In [ ]:
# ==============================================================================
# KROK 1: Generowanie realistycznego zbioru Churnu Klientów (Baza CRM)
# ==============================================================================
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (10, 5)
np.random.seed(42)

n_clients = 1200

# Cechy klientów
data = {
    'Wiek': np.random.randint(18, 70, size=n_clients),
    'Miesieczny_Rachunek': np.random.normal(120, 35, size=n_clients).round(2),
    'Staz_w_miesiacach': np.random.exponential(scale=24, size=n_clients).astype(int) + 1,
    'Liczba_Zgloszen_Pomocy': np.random.poisson(lam=1.2, size=n_clients),
    'Typ_Umowy': np.random.choice(['Miesieczna', 'Roczna', 'Dwuletnia'], size=n_clients, p=[0.5, 0.3, 0.2]),
    'Metoda_Platnosci': np.random.choice(['Karta', 'Przelew', 'Autopay'], size=n_clients)
}

df_churn = pd.DataFrame(data)

# Logika biznesowa generująca prawdopodobieństwo odejścia (Churn)
logit = (
    0.03 * df_churn['Miesieczny_Rachunek'] 
    + 0.8 * df_churn['Liczba_Zgloszen_Pomocy'] 
    - 0.08 * df_churn['Staz_w_miesiacach']
    + (df_churn['Typ_Umowy'] == 'Miesieczna') * 1.2
    - 3.5
)
prob = 1 / (1 + np.exp(-logit))
df_churn['Churn'] = (np.random.rand(n_clients) < prob).astype(int)

# Podsumowanie rozkładu klas
churn_rate = df_churn['Churn'].mean() * 100
print(f"Zbiór załadowany. Liczba klientów: {len(df_churn)}")
print(f"Odsetek odejść (Churn rate): {churn_rate:.1f}% (Typowy problem niezbalansowany w biznesie!)")
df_churn.head()

W realnym biznesie klasy niemal nigdy nie są rozłożone 50/50. 

W telekomunikacji czy SaaS tylko ok. 15–20% klientów odchodzi. 

Gdyby model po prostu zawsze mówił 'klient zostanie', miałby ponad 80% dokładności (Accuracy), ale z perspektywy zysków firmy byłby bezużyteczny.

In [ ]:
# ==============================================================================
# KROK 2: Podział danych z zachowaniem proporcji klas (Stratify) i Preprocessing
# ==============================================================================
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

X = df_churn.drop('Churn', axis=1)
y = df_churn['Churn']

# Stratify=y gwarantuje taki sam procent churnu w zbiorze Train i Test
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

num_features = ['Wiek', 'Miesieczny_Rachunek', 'Staz_w_miesiacach', 'Liczba_Zgloszen_Pomocy']
cat_features = ['Typ_Umowy', 'Metoda_Platnosci']

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), num_features),
        ('cat', OneHotEncoder(drop='first', sparse_output=False), cat_features)
    ]
)

print(f"Zbiór treningowy: {X_train.shape[0]} rekordów | Zbiór testowy: {X_test.shape[0]} rekordów")

Używamy parametru stratify=y, aby zbiór testowy odzwierciedlał dokładnie ten sam poziom ryzyka co treningowy. 

Budujemy czysty pipeline przekształcający zmienne kategoryczne i numeryczne.

In [ ]:
# ==============================================================================
# KROK 3: Model bazowy (Regresja Logistyczna) i zmiana progu odcięcia (Threshold)
# ==============================================================================
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix, classification_report

log_reg = Pipeline([
    ('prep', preprocessor),
    ('model', LogisticRegression(random_state=42))
])
log_reg.fit(X_train, y_train)

# Pobieramy czyste prawdopodobieństwa przynależności do klasy 1 (odejście)
y_probs = log_reg.predict_proba(X_test)[:, 1]

# Porównujemy dwa progi biznesowe: Domyślny (0.5) vs Proaktywny/Ostry (0.3)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for i, threshold in enumerate([0.5, 0.3]):
    y_pred_thresh = (y_probs >= threshold).astype(int)
    cm = confusion_matrix(y_test, y_pred_thresh)
    
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[i], cbar=False,
                xticklabels=['Zostaje (0)', 'Odchodzi (1)'],
                yticklabels=['Zostaje (0)', 'Odchodzi (1)'])
    axes[i].set_title(f"Macierz Pomyłek dla Progu = {threshold}\n"
                      f"Wykryty Churn (Recall): {cm[1,1]}/{cm[1,0]+cm[1,1]} "
                      f"({cm[1,1]/(cm[1,0]+cm[1,1])*100:.1f}%)")
    axes[i].set_xlabel("Predykcja Modelu")
    axes[i].set_ylabel("Faktyczny Stan Klienta")

plt.tight_layout()
plt.show()

Próg 0.5: 
- przegapiliśmy aż kilkunastu odchodzących klientów (False Negative w lewym dolnym rogu). 

Próg na 0.3 (prawa strona):
- łapiemy znacznie więcej odchodzących! 
 
Kosztem jest to, że wyślemy ofertę rabatową do kilku osób, które i tak by zostały (False Positive), ale w biznesie telekomunikacyjnym utrzymanie klienta jest 5x tańsze niż pozyskanie nowego.

In [ ]:
# ==============================================================================
# KROK 4: Porównanie modeli na Krzywej ROC (Niezależnie od wybranego progu)
# ==============================================================================
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import roc_curve, roc_auc_score

# Dodajemy drugi model bazowy: Naiwny Bayes (Syllabus pkt 3)
nb_model = Pipeline([
    ('prep', preprocessor),
    ('model', GaussianNB())
])
nb_model.fit(X_train, y_train)

# Obliczamy prawdopodobieństwa
y_probs_nb = nb_model.predict_proba(X_test)[:, 1]

# Wyznaczamy punkty krzywych ROC
fpr_lr, tpr_lr, _ = roc_curve(y_test, y_probs)
fpr_nb, tpr_nb, _ = roc_curve(y_test, y_probs_nb)

# Wyniki AUC
auc_lr = roc_auc_score(y_test, y_probs)
auc_nb = roc_auc_score(y_test, y_probs_nb)

plt.figure(figsize=(8, 6))
plt.plot(fpr_lr, tpr_lr, label=f'Regresja Logistyczna (AUC = {auc_lr:.3f})', lw=2.5, color='royalblue')
plt.plot(fpr_nb, tpr_nb, label=f'Naiwny Bayes (AUC = {auc_nb:.3f})', lw=2, color='darkorange')
plt.plot([0, 1], [0, 1], 'k--', alpha=0.5, label='Losowe zgadywanie (AUC = 0.500)')

plt.xlabel('False Positive Rate (Odsetek fałszywych alarmów)')
plt.ylabel('True Positive Rate (Czułość / Recall)')
plt.title('Krzywa ROC: Im bardziej wybrzuszona w lewy górny róg, tym lepszy model', fontsize=12)
plt.legend(loc='lower right')
plt.show()

Krzywa ROC testuje model na każdym możliwym progu od 0.0 do 1.0. 

Metryka AUC to pole pod tą krzywą. 
- Zwykły rzut monetą to 0.5, perfekcyjny model to 1.0. 

Regresja Logistyczna radzi sobie tutaj lepiej niż Naiwny Bayes.

In [ ]:
# ==============================================================================
# KROK 5: Wizualizacja struktury drzewa decyzyjnego (Wyjaśnialność biznesowa)
# ==============================================================================
from sklearn.tree import DecisionTreeClassifier, plot_tree

# Trenujemy płytkie drzewo o max_depth=3 (żeby zachować czytelność dla zarządu)
tree_model = Pipeline([
    ('prep', preprocessor),
    ('model', DecisionTreeClassifier(max_depth=3, criterion='gini', random_state=42))
])
tree_model.fit(X_train, y_train)

# Pobieramy nazwy zakodowanych cech
cat_encoded_names = list(tree_model.named_steps['prep'].named_transformers_['cat'].get_feature_names_out(cat_features))
all_feature_names = num_features + cat_encoded_names

plt.figure(figsize=(20, 8))
plot_tree(
    tree_model.named_steps['model'],
    feature_names=all_feature_names,
    class_names=['Zostaje', 'Odchodzi'],
    filled=True,
    rounded=True,
    fontsize=10
)
plt.title("Drzewo Decyzyjne (max_depth=3): Gotowa mapa reguł biznesowych (IF-THEN)", fontsize=14)
plt.show()

Biznes uwielbia drzewa. 
- Każdy węzeł to proste pytanie biznesowe: np. 'Czy liczba zgłoszeń do pomocy technicznej jest wysoka?'. 
- Kolor niebieski oznacza pewność odejścia, pomarańczowy pozostania. 

Ten wykres można wydrukować i dać dyrektorowi operacyjnemu bez linijki żargonu matematycznego.

In [ ]:
# ==============================================================================
# KROK 6: Potęga Modeli Zespołowych (Ensemble Learning)
# ==============================================================================
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier

models_ensemble = {
    "Pojedyncze Drzewo (Overfitted)": DecisionTreeClassifier(max_depth=None, random_state=42),
    "Random Forest (100 drzew - Bagging)": RandomForestClassifier(n_estimators=100, random_state=42),
    "Gradient Boosting (Boosting)": HistGradientBoostingClassifier(random_state=42)
}

results = []

for name, clf in models_ensemble.items():
    pipe = Pipeline([('prep', preprocessor), ('model', clf)])
    pipe.fit(X_train, y_train)
    
    # Ewaluacja na zbiorze treningowym i testowym
    train_auc = roc_auc_score(y_train, pipe.predict_proba(X_train)[:, 1])
    test_auc = roc_auc_score(y_test, pipe.predict_proba(X_test)[:, 1])
    
    results.append({
        'Model': name,
        'Train ROC-AUC': train_auc,
        'Test ROC-AUC': test_auc,
        'Różnica (Overfitting)': train_auc - test_auc
    })

df_results = pd.DataFrame(results).round(3)
print("--- PORÓWNANIE SKUTECZNOŚCI MODELI ---")
print(df_results.to_string(index=False))

Pojedyncze drzewo bez limitu głębokości osiąga na zbiorze treningowym perfekcyjne AUC = 1.000, ale na testowym drastycznie spada (Overfitting). 

Lasy Losowe i Gradient Boosting likwidują ten problem i osiągają najwyższe wyniki w świecie rzeczywistym.

In [ ]:
# ==============================================================================
# KROK 7: Które cechy najbardziej decydują o odejściu klienta?
# ==============================================================================
rf_fitted = RandomForestClassifier(n_estimators=100, random_state=42)
X_train_trans = preprocessor.fit_transform(X_train)
rf_fitted.fit(X_train_trans, y_train)

# Pobieramy ważność cech
importances = rf_fitted.feature_importances_
df_imp = pd.DataFrame({
    'Cecha': all_feature_names,
    'Waznosc': importances
}).sort_values('Waznosc', ascending=True)

plt.figure(figsize=(10, 6))
plt.barh(df_imp['Cecha'], df_imp['Waznosc'], color='teal')
plt.title("Ważność Cech wg Lasu Losowego (MDI Feature Importance)", fontsize=13)
plt.xlabel("Względny wpływ na decyzję modelu")
plt.tight_layout()
plt.show()

Oto odpowiedź dla zarządu: 
- Najważniejszym czynnikiem odejścia klienta jest liczba zgłoszeń do działu obsługi oraz staż w firmie. 
- Typ umowy miesięcznej również drastycznie podnosi ryzyko. 

Dzięki modelom zespołowym nie tylko precyzyjnie przewidujemy, kto odejdzie, ale też wiemy, co dział operacyjny musi naprawić w procesach firmy.